In [1]:
import tensorflow as tf
import numpy as np
from PIL import Image
import json
import faiss

In [2]:
IMG_SIZE = 224
CLASSES = ['CaS', 'CoS', 'Gum', 'MC', 'OC', 'OLP', 'OT']

MODEL_PATH = "teeth_case_based_model.h5"
FAISS_INDEX_PATH = "oral_cases.index"
LABELS_PATH = "case_labels.npy"
IDS_PATH = "case_ids.json"

In [4]:
IMAGE_PATH = "photos/Commissural Stomatitis.jpg"
TOP_K = 3

In [5]:
print(f"Loading model from {MODEL_PATH}...")
model = tf.keras.models.load_model(MODEL_PATH)

Loading model from teeth_case_based_model.h5...


In [6]:
embedding_model = tf.keras.Model(
    inputs=model.input,
    outputs=model.get_layer("embedding").output
)

print("Embedding model created ")


Embedding model created 


In [7]:
print(f"Loading FAISS index from {FAISS_INDEX_PATH}...")
index = faiss.read_index(FAISS_INDEX_PATH)

print(f"Loading labels from {LABELS_PATH}...")
case_labels = np.load(LABELS_PATH)

print(f"Loading ids from {IDS_PATH}...")
with open(IDS_PATH, "r") as f:
    case_ids = json.load(f)

print("FAISS database loaded successfully ")

Loading FAISS index from oral_cases.index...
Loading labels from case_labels.npy...
Loading ids from case_ids.json...
FAISS database loaded successfully 


In [8]:
img = Image.open(IMAGE_PATH).convert("RGB")
img = img.resize((IMG_SIZE, IMG_SIZE))
img_array = np.array(img) / 255.0
img_array = np.expand_dims(img_array, axis=0)


In [9]:
prediction = model.predict(img_array, verbose=0)

predicted_class_idx = np.argmax(prediction[0])
predicted_class = CLASSES[predicted_class_idx]
confidence = float(prediction[0][predicted_class_idx])

In [10]:
query_embedding = embedding_model.predict(img_array, verbose=0).astype("float32")


In [11]:
faiss.normalize_L2(query_embedding)


In [12]:
similarity_scores, indices = index.search(query_embedding, TOP_K)


In [13]:
similar_cases = []

for rank, idx in enumerate(indices[0]):
    similar_cases.append({
        "case_id": case_ids[idx],
        "diagnosis": CLASSES[case_labels[idx]],
        "similarity": float(similarity_scores[0][rank]),
        "similarity_percent": float(similarity_scores[0][rank] * 100)
    })

In [14]:
same_diag_count = 0
for c in similar_cases:
    if c["diagnosis"] == predicted_class:
        same_diag_count += 1

print("\n==================== RESULT ====================")
print(f"Image: {IMAGE_PATH}")
print(f"Predicted Disease: {predicted_class}")
print(f"Confidence: {confidence:.4f}")

print("\nTop Similar Cases:")
for c in similar_cases:
    print(f"- Case ID: {c['case_id']} | Similarity: {c['similarity_percent']:.2f}% | Diagnosis: {c['diagnosis']}")

print("\nReasoning:")
print(f"{same_diag_count} out of {len(similar_cases)} of the most similar cases had the same diagnosis as the predicted one.")



==================== RESULT ====================
Image: photos/Commissural Stomatitis.jpg
Predicted Disease: CoS
Confidence: 0.9990

Top Similar Cases:
- Case ID: CaS\a_142_0_2483.jpg | Similarity: 92.86% | Diagnosis: CoS
- Case ID: OT\ot_1215_0_7738.jpg | Similarity: 92.61% | Diagnosis: CoS
- Case ID: Gum\g_1228_0_5955.jpg | Similarity: 92.28% | Diagnosis: CoS

Reasoning:
3 out of 3 of the most similar cases had the same diagnosis as the predicted one.


In [15]:
print("\nAll predictions:")
for i, cls in enumerate(CLASSES):
    print(f"  {cls}: {float(prediction[0][i]):.4f}")


All predictions:
  CaS: 0.0000
  CoS: 0.9990
  Gum: 0.0010
  MC: 0.0000
  OC: 0.0000
  OLP: 0.0000
  OT: 0.0000
